# 01 - Data Preprocessing

## Objective
To clean and transform data, in this stage we'll handle duplicates and split features and targets.

In [77]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import joblib

# Loading dataset 

In [78]:
data_path = Path("../data/creditcard.csv")
df = pd.read_csv(data_path)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [79]:
print("Original dataset shape:", df.shape)

Original dataset shape: (284807, 31)


In [80]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)

Duplicate rows found: 1081


# Duplicates Handling Methods: Remove
- We will first remove duplicates, for us to to have clean data, althogh this may remove repeated transactions, it prevents model bias, overfitting, and reduce generalization.

###### Other information not really necessary group disscusion on
- Because our dataset represent more of real world scnario we will also keep duplicates.

- The final decision will be made based on model performance

In [81]:
df = df.drop_duplicates(keep='first')
print("Dataset shape after removing duplicates:", df.shape)


Dataset shape after removing duplicates: (283726, 31)


Duplicates were removed to prevent bias during model training. Keeping duplicates could cause the model to overfit repeated patterns, especially given the already imbalanced dataset.

## SPlit Features and Target

In [82]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


Feature shape: (283726, 30)
Target shape: (283726,)


# Train/Test Split

In [83]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

X_train shape: (226980, 30), y_train shape: (226980,)
X_test shape: (56746, 30), y_test shape: (56746,)


# Class Distribution

In [84]:
train_distribution = y_train.value_counts(normalize=True) * 100
test_distribution =y_test.value_counts(normalize=True) * 100

print(f"Training class distribution:{train_distribution}")
print(f"\nTesting class distribution:{test_distribution}")

Training class distribution:Class
0    99.833466
1     0.166534
Name: proportion, dtype: float64

Testing class distribution:Class
0    99.832587
1     0.167413
Name: proportion, dtype: float64


# Scale Time and Amount

In [85]:
scaler = StandardScaler()

X_train = X_train.copy()
X_test = X_test.copy()

X_train[["Time", "Amount"]] = scaler.fit_transform(X_train[["Time", "Amount"]])
X_test[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

In [86]:
X_train[["Time", "Amount"]].describe()

,Time,Amount
count,2.269800e+05,2.269800e+05
mean,-1.522636e-16,7.487965e-17
std,1.000002e+00,1.000002e+00
min,-1.998400e+00,-3.596386e-01
25%,-8.561822e-01,-3.364866e-01
50%,-2.123526e-01,-2.697972e-01
75%,9.363191e-01,-4.287453e-02
max,1.640237e+00,7.962087e+01


In [89]:
Output_path = Path("../outputs/results")


In [90]:
joblib.dump({ "X_train" : X_train,
        "X_test" : X_test,
        "y_train" : y_train,
        "y_test" : y_test,
        "scaler" : scaler
        }, Output_path / "creditcard_cleaned.pk1"
       
)

print("processed data saved successfully.")

processed data saved successfully.


## Preprocessing Summary

- Duplicate rows were removed from the dataset.
- Features and target were separated.
- A stratified train/test split was used to preserve the fraud-to-normal transaction ratio.
- Only `Time` and `Amount` were scaled because `V1` to `V28` are already PCA-transformed.
- The scaler was fitted only on the training data to avoid data leakage.
- The processed data was saved for use in the baseline modelling notebook.